In [2]:
import pandas as pd
import statistics as st
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("../data/raw/train.csv")

In [3]:
df["IsFemale"] = df["Sex"] == "female"
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Build Title and finish ALL text-based work on it first
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df.loc[~df["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Title"] = "Rare"

# Fill Age while Title is still plain text (needed for groupby)
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))

# Fill Embarked before encoding it
df["Embarked"] = df["Embarked"].fillna("S")

# NOW one-hot encode both, once everything that needed the plain versions is done
df = pd.get_dummies(df, columns=["Title"], prefix="Title")
df = pd.get_dummies(df, columns=["Embarked"], prefix="Embarked")

# HasCabin doesn't depend on ordering relative to the above
df["HasCabin"] = df["Cabin"].notna()

In [4]:
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S"
]
X = df[feature_cols]
y = df["Survived"]

# 11. Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [5]:
X.isna().sum()

IsFemale        0
Pclass          0
Age             0
Fare            0
FamilySize      0
HasCabin        0
Title_Master    0
Title_Miss      0
Title_Mr        0
Title_Mrs       0
Title_Rare      0
Embarked_C      0
Embarked_Q      0
Embarked_S      0
dtype: int64

In [6]:
X.shape

(891, 14)

In [7]:
X.head()

,IsFemale,Pclass,Age,Fare,FamilySize,HasCabin,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Embarked_C,Embarked_Q,Embarked_S
0,False,3,22.0,7.2500,2,False,False,False,True,False,False,False,False,True
1,True,1,38.0,71.2833,2,True,False,False,False,True,False,True,False,False
2,True,3,26.0,7.9250,1,False,False,True,False,False,False,False,False,True
3,True,1,35.0,53.1000,2,True,False,False,False,True,False,False,False,True
4,False,3,35.0,8.0500,1,False,False,False,True,False,False,False,False,True


In [8]:
from sklearn.tree import DecisionTreeClassifier

tree_model0 = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model0.fit(X_train, y_train)

tree_predictions = tree_model0.predict(X_val)
accuracy_score(y_val, tree_predictions)

0.8156424581005587

In [9]:
from sklearn.tree import DecisionTreeClassifier

random_states = [1, 7, 42, 55, 99]
depths_to_try = [2, 3, 4, 5, 6, 8]
results = {}

for depth in depths_to_try:
    scores = []
    for rs in random_states:
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=rs
        )
        tree_model = DecisionTreeClassifier(max_depth=depth, random_state=rs)
        tree_model.fit(X_train, y_train)
        acc = accuracy_score(y_val, tree_model.predict(X_val))
        scores.append(acc)
    results[depth] = scores
    print(f"depth={depth}: mean={sum(scores)/len(scores)*100:.2f}%, min={min(scores)*100:.2f}%, max={max(scores)*100:.2f}%")

depth=2: mean=79.22%, min=76.54%, max=83.24%
depth=3: mean=81.90%, min=78.21%, max=84.36%
depth=4: mean=81.79%, min=80.45%, max=83.24%
depth=5: mean=80.00%, min=78.21%, max=81.56%
depth=6: mean=80.45%, min=79.33%, max=81.56%
depth=8: mean=80.00%, min=78.77%, max=81.56%


In [10]:
importances = pd.Series(tree_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

Title_Mr        0.409223
Fare            0.213974
Age             0.158503
Pclass          0.080531
FamilySize      0.045473
Title_Rare      0.040816
IsFemale        0.020982
HasCabin        0.016974
Title_Miss      0.005079
Title_Mrs       0.004575
Embarked_C      0.003870
Title_Master    0.000000
Embarked_Q      0.000000
Embarked_S      0.000000
dtype: float64

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

importances = pd.Series(tree_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

Title_Mr        0.637112
Pclass          0.157287
FamilySize      0.083419
HasCabin        0.050947
Title_Rare      0.050510
Age             0.013116
Fare            0.007608
IsFemale        0.000000
Title_Miss      0.000000
Title_Master    0.000000
Title_Mrs       0.000000
Embarked_C      0.000000
Embarked_Q      0.000000
Embarked_S      0.000000
dtype: float64

In [12]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
forest_model.fit(X_train, y_train)

forest_predictions = forest_model.predict(X_val)
accuracy_score(y_val, forest_predictions)

0.7988826815642458

In [13]:
random_states = [1, 7, 42, 55, 99]
scores_forest = []

for rs in random_states:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
    forest_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=rs)
    forest_model.fit(X_train, y_train)
    acc = accuracy_score(y_val, forest_model.predict(X_val))
    scores_forest.append(acc)

print(scores_forest)

[0.8324022346368715, 0.8044692737430168, 0.7988826815642458, 0.8435754189944135, 0.8547486033519553]


In [14]:
importances = pd.Series(forest_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

Title_Mr        0.304396
IsFemale        0.220352
Fare            0.091454
Pclass          0.082332
Title_Mrs       0.081884
Title_Miss      0.055284
HasCabin        0.050019
FamilySize      0.047257
Age             0.035611
Title_Master    0.009425
Embarked_S      0.009016
Embarked_C      0.009001
Title_Rare      0.002019
Embarked_Q      0.001951
dtype: float64

In [15]:
max(scores_forest)

0.8547486033519553

In [16]:
random_states = [1, 7, 42, 55, 99]
depths_to_try = [3, 5, 7, None]   # None = no depth limit at all
n_trees_to_try = [50, 100, 200]

for depth in depths_to_try:
    for n_trees in n_trees_to_try:
        scores = []
        for rs in random_states:
            X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
            m = RandomForestClassifier(n_estimators=n_trees, max_depth=depth, random_state=rs)
            m.fit(X_train, y_train)
            scores.append(accuracy_score(y_val, m.predict(X_val)))
        print(f"depth={depth}, n_trees={n_trees}: mean={st.mean(scores)*100:.2f}%, stdev={st.stdev(scores)*100:.2f}")

depth=3, n_trees=50: mean=82.79%, stdev=2.25
depth=3, n_trees=100: mean=82.68%, stdev=2.44
depth=3, n_trees=200: mean=82.35%, stdev=2.19
depth=5, n_trees=50: mean=83.02%, stdev=1.75
depth=5, n_trees=100: mean=83.80%, stdev=1.89
depth=5, n_trees=200: mean=84.25%, stdev=2.45
depth=7, n_trees=50: mean=82.57%, stdev=2.54
depth=7, n_trees=100: mean=82.79%, stdev=1.99
depth=7, n_trees=200: mean=83.13%, stdev=1.45
depth=None, n_trees=50: mean=79.66%, stdev=1.16
depth=None, n_trees=100: mean=80.11%, stdev=1.29
depth=None, n_trees=200: mean=80.00%, stdev=0.73


In [17]:
ticket_counts = df["Ticket"].value_counts()
df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)

In [18]:
df[["FamilySize", "TicketGroupSize"]].describe()

,FamilySize,TicketGroupSize
count,891.000000,891.000000
mean,1.904602,1.787879
std,1.613459,1.361142
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,11.000000,7.000000


In [19]:
(df["FamilySize"] == df["TicketGroupSize"]).mean()

np.float64(0.6767676767676768)

In [20]:
mismatch = df[df["TicketGroupSize"] != df["FamilySize"]]
mismatch[["Name", "SibSp", "Parch", "FamilySize", "Ticket", "TicketGroupSize"]].sort_values("TicketGroupSize", ascending=False).head(15)

,Name,SibSp,Parch,FamilySize,Ticket,TicketGroupSize
838,"Chip, Mr. Chang",0,0,1,1601,7
792,"Sage, Miss. Stella Anna",8,2,11,CA. 2343,7
826,"Lam, Mr. Len",0,0,1,1601,7
169,"Ling, Mr. Lee",0,0,1,1601,7
159,"Sage, Master. Thomas Henry",8,2,11,CA. 2343,7
180,"Sage, Miss. Constance Gladys",8,2,11,CA. 2343,7
201,"Sage, Mr. Frederick",8,2,11,CA. 2343,7
74,"Bing, Mr. Lee",0,0,1,1601,7
509,"Lang, Mr. Fang",0,0,1,1601,7
692,"Lam, Mr. Ali",0,0,1,1601,7


In [21]:
test_df = pd.read_csv("../data/raw/test.csv")

all_tickets = pd.concat([df["Ticket"], test_df["Ticket"]])

In [22]:
combined_ticket_counts = all_tickets.value_counts()
df["TicketGroupSize"] = df["Ticket"].map(combined_ticket_counts)

In [23]:
df[df["Name"].str.contains("Sage")][["Name", "FamilySize", "TicketGroupSize"]]

,Name,FamilySize,TicketGroupSize
159,"Sage, Master. Thomas Henry",11,11
180,"Sage, Miss. Constance Gladys",11,11
201,"Sage, Mr. Frederick",11,11
324,"Sage, Mr. George John Jr",11,11
641,"Sagesser, Mlle. Emma",1,2
792,"Sage, Miss. Stella Anna",11,11
846,"Sage, Mr. Douglas Bullen",11,11
863,"Sage, Miss. Dorothy Edith ""Dolly""",11,11


In [24]:
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S",
    "TicketGroupSize"
]
X = df[feature_cols]
y = df["Survived"]

In [25]:
import statistics as st

random_states = [1, 7, 42, 55, 99]
scores_ticket = []

for rs in random_states:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
    m = LogisticRegression(max_iter=1000)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_val, m.predict(X_val))
    scores_ticket.append(acc)

print(scores_ticket)
print(f"mean={st.mean(scores_ticket)*100:.2f}%, stdev={st.stdev(scores_ticket)*100:.2f}")

[0.8268156424581006, 0.8212290502793296, 0.8324022346368715, 0.8379888268156425, 0.8659217877094972]
mean=83.69%, stdev=1.74


In [26]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5)
print(scores)
print(f"mean={scores.mean()*100:.2f}%, std={scores.std()*100:.2f}")

[0.84357542 0.82022472 0.79213483 0.82022472 0.87078652]
mean=82.94%, std=2.63


In [27]:
from sklearn.model_selection import StratifiedKFold

cv_shuffled = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_shuffled = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv_shuffled)
print(scores_shuffled)
print(f"mean={scores_shuffled.mean()*100:.2f}%, std={scores_shuffled.std()*100:.2f}")

[0.84916201 0.81460674 0.81460674 0.84269663 0.83707865]
mean=83.16%, std=1.44


In [28]:
depths = [2, 3, 4, 6, 8]

for d in depths:
    scores = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42), X, y, cv=cv_shuffled)
    print(f"depth={d}: mean={scores.mean()*100:.2f}%, std={scores.std()*100:.2f}")

depth=2: mean=77.89%, std=2.42
depth=3: mean=81.59%, std=1.73
depth=4: mean=82.27%, std=1.07
depth=6: mean=81.93%, std=2.71
depth=8: mean=82.71%, std=1.59


In [29]:
num_trees = [50, 100, 500]

for n in num_trees:
    for d in depths:
        scores = cross_val_score(RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42), X, y, cv=cv_shuffled)
        print(f"trees={n}, depth={d}: mean={scores.mean()*100:.2f}%, std={scores.std()*100:.2f}")

trees=50, depth=2: mean=80.81%, std=1.94
trees=50, depth=3: mean=82.49%, std=0.75
trees=50, depth=4: mean=83.28%, std=0.44
trees=50, depth=6: mean=83.95%, std=0.93
trees=50, depth=8: mean=83.61%, std=1.12
trees=100, depth=2: mean=80.14%, std=2.36
trees=100, depth=3: mean=82.72%, std=0.98
trees=100, depth=4: mean=83.50%, std=0.48
trees=100, depth=6: mean=83.95%, std=1.07
trees=100, depth=8: mean=84.40%, std=0.92
trees=500, depth=2: mean=79.80%, std=2.31
trees=500, depth=3: mean=82.94%, std=0.77
trees=500, depth=4: mean=83.61%, std=0.57
trees=500, depth=6: mean=83.50%, std=1.18
trees=500, depth=8: mean=83.95%, std=1.29


In [30]:
depths_extended = [8, 10, 12, 15, 20]

for d in depths_extended:
    scores = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42), X, y, cv=cv_shuffled)
    print(f"depth={d}: mean={scores.mean()*100:.2f}%, std={scores.std()*100:.2f}")

depth=8: mean=84.40%, std=0.92
depth=10: mean=84.06%, std=1.55
depth=12: mean=83.61%, std=1.40
depth=15: mean=82.49%, std=0.45
depth=20: mean=82.15%, std=1.11
